In [1]:
import os
from pathlib import Path
import jwst
print(jwst.__version__)
from jwst import datamodels
from jwst.datamodels import dqflags

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
# import natural units:
import natural_units as nu
# multi-core/thread:
import concurrent.futures
import re


1.19.2


In [2]:
# 全局变量，一波定义完
number_of_electron_bins = 30
# bin 的范围。用 range(dn_min, dn_max+1) to include dn_max
dn_min = -200
dn_max = 400

# define the mass and cross section grid we are working with.
log_m_min  = -3
log_m_max  = 1
n_m    = 33   # From 1e-3  to 10 GeV
#log cs shift from the balloon line
log_cs_min = -3
log_cs_max = 0
n_cs   = 49
m_grid  = np.logspace(log_m_min, log_m_max, n_m)
cs_grid = np.logspace(log_cs_min, log_cs_max, n_cs)
center_line = np.array([2.15504637e-23, 1.88334907e-23, 1.53030461e-23, 1.32738030e-23,
       1.26359147e-23, 1.37889395e-23, 1.61669130e-23, 1.95514385e-23,
       2.40008514e-23, 3.14137096e-23, 4.03532201e-23, 5.19960700e-23,
       6.95747264e-23, 9.69011074e-23, 1.40957345e-22, 2.05768490e-22,
       2.84518575e-22, 3.82210762e-22, 5.17381239e-22, 7.29305911e-22,
       1.08351297e-21, 1.55439713e-21, 2.15203017e-21, 3.00211451e-21,
       4.12016417e-21, 5.67974728e-21, 7.78952222e-21, 1.06757849e-20,
       1.45615810e-20, 1.99263396e-20, 2.70914228e-20, 3.67600191e-20,
       4.94000000e-20])

# fractions to run
fractions = ['4e-3', '1e-3', '5e-4', '2e-4', '1e-4']

In [3]:
#样本数量
sample_size = int(1e8)
# Generate pixel_value_raw by Poisson distribution:
def generate_raw_value(binned_signals):
    raw_value = np.zeros(sample_size)
    for i in range(len(binned_signals)):
        lambda_param = binned_signals[i]
        poisson_samples = np.random.poisson(lambda_param, sample_size)
        raw_value += (i+1) * poisson_samples
    return raw_value

binned_signal_path = Path('../data/binned_signal_w_shield_w_lindhard/')

def process_file(filename, frac):
    frac_rescale = float(frac) * 100 / 0.4
    file_stem = filename.stem
    pattern = r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2"
    # 使用 re.search() 匹配并提取
    match = re.search(pattern, file_stem)
    mDM_in_GeV = float(match.group(1))  # 第一个括号组匹配 mDM 的值
    sigma_e_in_cm2 = float(match.group(2))  # 第二个括号组匹配 sigma 的值
    m_index = np.argmin(np.abs(m_grid - mDM_in_GeV))
    cs_index = np.argmin(np.abs(cs_grid * center_line[m_index] - sigma_e_in_cm2))

    binned_signals = np.loadtxt(filename) * frac_rescale
    dm_sample = generate_raw_value(binned_signals)
    dm_poisson_counts, bin_edges = np.histogram(dm_sample, bins=range(dn_min, dn_max+1))
    dm_poisson_counts = dm_poisson_counts / sample_size   # to get PDF
    np.savetxt('./results_w_lin/DM_binned_shielding_frac_'+ frac +'/'+ str(m_index) + '_' + str(cs_index) +'.txt', dm_poisson_counts)
    print(file_stem[15:] + '  completed,     index' + str(m_index) + '_' + str(cs_index))

    return 0

def is_original_grid_file(filename):
    if filename.suffix != '.txt':
        return False
    match = re.search(r"mDM=([0-9.eE+-]+)_GeV_sigma=([0-9.eE+-]+)_cm2", filename.stem)
    if match is None:
        return False
    mDM_in_GeV = float(match.group(1))
    sigma_e_in_cm2 = float(match.group(2))
    m_index = np.argmin(np.abs(m_grid - mDM_in_GeV))
    min_sigma = center_line[m_index] * 10**log_cs_min
    max_sigma = center_line[m_index] * 10**log_cs_max
    return min_sigma * (1 - 1e-5) <= sigma_e_in_cm2 <= max_sigma * (1 + 1e-5)

binned_signal_files = [file for file in binned_signal_path.iterdir() if is_original_grid_file(file)]
if len(binned_signal_files) != n_m * n_cs:
    raise RuntimeError(f'Expected original-grid input files, found {len(binned_signal_files)}.')

In [4]:
for frac in fractions:
    # 使用 ProcessPoolExecutor 并行处理文件，限制最大进程数为 10
    with concurrent.futures.ProcessPoolExecutor(max_workers=16) as executor:
        # 获取文件夹中所有的文件路径
        file_paths = binned_signal_files
        
        # 提交文件处理任务到进程池
        futures = {executor.submit(process_file, file, frac): file for file in file_paths}
        
        # 逐个处理完成的任务
        for future in concurrent.futures.as_completed(futures):
            file = futures[future]
            try:
                result = future.result()  # 获取任务的返回值
                print(f"Finished processing {result}")
            except Exception as exc:
                print(f"Error processing {file}: {exc}")

mDM=0.177828_GeV_sigma=3.87981e-24_cm2  completed,     index18_14
Finished processing 0
mDM=0.00316228_GeV_sigma=1.68503e-24_cm2  completed,     index4_34
Finished processing 0
mDM=3.16228_GeV_sigma=1.26098e-22_cm2  completed,     index28_15
Finished processing 0
mDM=10_GeV_sigma=7.60724e-21_cm2  completed,     index32_35
Finished processing 0
mDM=0.749894_GeV_sigma=7.11914e-22_cm2  completed,     index23_38
Finished processing 0
mDM=1.77828_GeV_sigma=6.74545e-22_cm2  completed,     index26_31
Finished processing 0
mDM=0.00237137_GeV_sigma=3.63492e-25_cm2  completed,     index3_23
Finished processing 0
mDM=0.316228_GeV_sigma=5.27636e-23_cm2  completed,     index20_27
Finished processing 0
mDM=1.33352_GeV_sigma=1.01002e-23_cm2  completed,     index25_4
Finished processing 0
mDM=0.0749894_GeV_sigma=2.37618e-23_cm2  completed,     index15_33
Finished processing 0
mDM=0.00237137_GeV_sigma=1.77009e-26_cm2  completed,     index3_2
Finished processing 0
mDM=4.21697_GeV_sigma=1.29398e-21_cm2  